In [1]:
import numpy as np
import xarray as xr
import scipy.stats as stats

In [3]:
import inspect

import numpy as np
import pandas as pd
import xarray as xr
import scipy.stats as stats


def synthetic_skewed_sample(size=501, seed=0, sigma=1.0, nan_fraction=0.0):
    rng = np.random.default_rng(seed)
    sample = rng.lognormal(mean=0.0, sigma=sigma, size=size)
    if nan_fraction:
        nan_count = int(round(size * nan_fraction))
        if nan_count:
            nan_index = rng.choice(size, size=nan_count, replace=False)
            sample[nan_index] = np.nan
    return sample


def dataarray_quantile_median(data, method_name):
    array = xr.DataArray(data, dims="sample")
    quantile_signature = inspect.signature(xr.DataArray.quantile)
    quantile_kwargs = {"q": 0.5, "dim": "sample"}
    if "method" in quantile_signature.parameters:
        quantile_kwargs["method"] = method_name
    else:
        quantile_kwargs["interpolation"] = method_name
    return float(array.quantile(**quantile_kwargs).item())


methods = ["linear", "lower", "higher", "midpoint", "nearest"]
scenarios = [
    {"size": 51, "sigma": 0.5, "nan_fraction": 0.0},
    {"size": 501, "sigma": 1.0, "nan_fraction": 0.0},
    {"size": 5001, "sigma": 1.5, "nan_fraction": 0.0},
    {"size": 501, "sigma": 1.0, "nan_fraction": 0.05},
]

results = []
for scenario_index, scenario in enumerate(scenarios, start=1):
    sample = synthetic_skewed_sample(seed=42 + scenario_index, **scenario)
    reference = float(np.nanmedian(sample))
    hdmedian = float(stats.mstats.hdmedian(sample))

    for method_name in methods:
        try:
            quantile_median = dataarray_quantile_median(sample, method_name)
            quantile_error = ""
        except Exception as error:
            quantile_median = np.nan
            quantile_error = repr(error)

        results.append(
            {
                "scenario": scenario_index,
                "size": scenario["size"],
                "sigma": scenario["sigma"],
                "nan_fraction": scenario["nan_fraction"],
                "method": method_name,
                "dataarray_quantile_median": quantile_median,
                "hdmedian": hdmedian,
                "reference_nanmedian": reference,
                "delta_vs_reference": quantile_median - reference if np.isfinite(quantile_median) else np.nan,
                "hdmedian_delta_vs_reference": hdmedian - reference,
                "quantile_error": quantile_error,
            }
        )

results_frame = pd.DataFrame(results)
results_frame.sort_values(["scenario", "method"])

/glade/derecho/scratch/bbuchovecky/tmp/ipykernel_4943/463577808.py:43: UserWarning: Warning: converting a masked element to nan.
  hdmedian = float(stats.mstats.hdmedian(sample))


,scenario,size,sigma,nan_fraction,method,dataarray_quantile_median,hdmedian,reference_nanmedian,delta_vs_reference,hdmedian_delta_vs_reference,quantile_error
2,1,51,0.5,0.00,higher,1.004648,0.952287,1.004648,0.000000,-0.052362,
0,1,51,0.5,0.00,linear,1.004648,0.952287,1.004648,0.000000,-0.052362,
1,1,51,0.5,0.00,lower,1.004648,0.952287,1.004648,0.000000,-0.052362,
3,1,51,0.5,0.00,midpoint,1.004648,0.952287,1.004648,0.000000,-0.052362,
4,1,51,0.5,0.00,nearest,1.004648,0.952287,1.004648,0.000000,-0.052362,
7,2,501,1.0,0.00,higher,1.069384,1.076693,1.069384,0.000000,0.007308,
5,2,501,1.0,0.00,linear,1.069384,1.076693,1.069384,0.000000,0.007308,
6,2,501,1.0,0.00,lower,1.069384,1.076693,1.069384,0.000000,0.007308,
8,2,501,1.0,0.00,midpoint,1.069384,1.076693,1.069384,0.000000,0.007308,
9,2,501,1.0,0.00,nearest,1.069384,1.076693,1.069384,0.000000,0.007308,
